In [ ]:
import os, time, random, math
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
from obelix import OBELIX

device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

STATE_DIM=18
N_ACTIONS=5
ACTIONS=("L45","L22","FW","R22","R45")

GRU_HIDDEN=128
FC_EMBED=64

LR=2.5e-4
GAMMA=0.99
GAE_LAMBDA=0.95
CLIP_EPS=0.2
ENTROPY_C=0.01
VF_COEF=0.5
MAX_GRAD=0.5
N_EPOCHS=4
CHUNK_LEN=32
ROLLOUT_LEN=512

TOTAL_STEPS=150000
SAVE_PATH="rppo_weights.pth"

ARENA=500
SCALE=5
MAX_EP=1000
WALL_OBSTACLES=True
BOX_SPEED=2
BASE_SEED=42

DIFFICULTY_WEIGHTS={0:0.3,2:0.3,3:0.4}

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed=nn.Sequential(
            nn.Linear(STATE_DIM,FC_EMBED),
            nn.Tanh(),
            nn.Linear(FC_EMBED,FC_EMBED),
            nn.Tanh()
        )
        self.gru=nn.GRUCell(FC_EMBED,GRU_HIDDEN)
        self.actor=nn.Sequential(
            nn.Linear(GRU_HIDDEN,FC_EMBED),
            nn.Tanh(),
            nn.Linear(FC_EMBED,N_ACTIONS)
        )
        self.critic=nn.Sequential(
            nn.Linear(GRU_HIDDEN,FC_EMBED),
            nn.Tanh(),
            nn.Linear(FC_EMBED,1)
        )
        for m in self.modules():
            if isinstance(m,nn.Linear):
                nn.init.orthogonal_(m.weight,gain=math.sqrt(2))
                nn.init.zeros_(m.bias)
        nn.init.orthogonal_(self.actor[-1].weight,gain=0.01)
        nn.init.orthogonal_(self.critic[-1].weight,gain=1.0)

    def forward(self,obs,h):
        e=self.embed(obs)
        h_new=self.gru(e,h)
        logits=self.actor(h_new)
        val=self.critic(h_new).squeeze(-1)
        return logits,val,h_new

    def init_hidden(self,batch=1):
        return torch.zeros(batch,GRU_HIDDEN,device=device)

def compute_gae(rewards,values,dones,last_val):
    n=len(rewards)
    adv=np.zeros(n,dtype=np.float32)
    gae=0.0
    for t in reversed(range(n)):
        nxt=last_val if t==n-1 else values[t+1]
        mask=1.0-dones[t]
        delta=rewards[t]+GAMMA*nxt*mask-values[t]
        gae=delta+GAMMA*GAE_LAMBDA*mask*gae
        adv[t]=gae
    return adv,adv+values

_diffs=list(DIFFICULTY_WEIGHTS.keys())
_probs=np.array([DIFFICULTY_WEIGHTS[d] for d in _diffs])
_probs=_probs/_probs.sum()

def sample_diff():
    return int(np.random.choice(_diffs,p=_probs))

def make_env(diff,seed):
    return OBELIX(
        scaling_factor=SCALE,
        arena_size=ARENA,
        max_steps=MAX_EP,
        wall_obstacles=WALL_OBSTACLES,
        difficulty=diff,
        box_speed=BOX_SPEED,
        seed=seed
    )

def main():
    net=Net().to(device)
    opt=optim.Adam(net.parameters(),lr=LR)

    buf_obs=np.zeros((ROLLOUT_LEN,STATE_DIM),dtype=np.float32)
    buf_act=np.zeros(ROLLOUT_LEN,dtype=np.int64)
    buf_logp=np.zeros(ROLLOUT_LEN,dtype=np.float32)
    buf_val=np.zeros(ROLLOUT_LEN,dtype=np.float32)
    buf_rew=np.zeros(ROLLOUT_LEN,dtype=np.float32)
    buf_done=np.zeros(ROLLOUT_LEN,dtype=np.float32)
    buf_h=np.zeros((ROLLOUT_LEN,GRU_HIDDEN),dtype=np.float32)

    diff=sample_diff()
    env=make_env(diff,BASE_SEED)
    obs=env.reset(seed=BASE_SEED)
    h=net.init_hidden(1)

    total_steps=0
    ep=0

    while total_steps<TOTAL_STEPS:
        net.eval()
        with torch.no_grad():
            for t in range(ROLLOUT_LEN):
                obs_t=torch.FloatTensor(obs).unsqueeze(0).to(device)
                logits,val,h_new=net(obs_t,h)
                dist=Categorical(logits=logits)
                action=dist.sample()
                logp=dist.log_prob(action)

                act_i=action.item()
                obs_next,rew,done=env.step(ACTIONS[act_i],render=False)

                buf_obs[t]=obs
                buf_act[t]=act_i
                buf_logp[t]=logp.item()
                buf_val[t]=val.item()
                buf_rew[t]=rew
                buf_done[t]=float(done)
                buf_h[t]=h.squeeze(0).cpu().numpy()

                obs=obs_next
                h=h_new
                total_steps+=1

                if done:
                    ep+=1
                    diff=sample_diff()
                    env=make_env(diff,BASE_SEED+ep)
                    obs=env.reset(seed=BASE_SEED+ep)
                    h=net.init_hidden(1)

        with torch.no_grad():
            obs_t=torch.FloatTensor(obs).unsqueeze(0).to(device)
            _,last_v,_=net(obs_t,h)
            last_v=last_v.item()

        adv,ret=compute_gae(buf_rew,buf_val,buf_done,last_v)

        net.train()
        adv_t=torch.FloatTensor(adv).to(device)
        adv_t=(adv_t-adv_t.mean())/(adv_t.std()+1e-8)
        ret_t=torch.FloatTensor(ret).to(device)
        logp_old=torch.FloatTensor(buf_logp).to(device)
        act_t=torch.LongTensor(buf_act).to(device)

        for _ in range(N_EPOCHS):
            for s in range(0,ROLLOUT_LEN-CHUNK_LEN,CHUNK_LEN):
                h_chunk=torch.FloatTensor(buf_h[s]).unsqueeze(0).to(device)

                new_logps=[]
                new_vals=[]
                new_ents=[]

                for t in range(s,s+CHUNK_LEN):
                    obs_t=torch.FloatTensor(buf_obs[t]).unsqueeze(0).to(device)
                    logits,val,h_chunk=net(obs_t,h_chunk)
                    dist=Categorical(logits=logits)
                    a=act_t[t].unsqueeze(0)
                    new_logps.append(dist.log_prob(a).squeeze())
                    new_vals.append(val.squeeze())
                    new_ents.append(dist.entropy().squeeze())

                new_logps=torch.stack(new_logps)
                new_vals=torch.stack(new_vals)
                new_ents=torch.stack(new_ents)

                sl=slice(s,s+CHUNK_LEN)
                ratio=(new_logps-logp_old[sl]).exp()
                clip_r=torch.clamp(ratio,1-CLIP_EPS,1+CLIP_EPS)

                actor=-torch.min(ratio*adv_t[sl],clip_r*adv_t[sl]).mean()
                critic=nn.functional.mse_loss(new_vals,ret_t[sl])
                entropy=-new_ents.mean()

                loss=actor+VF_COEF*critic+ENTROPY_C*entropy

                opt.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(net.parameters(),MAX_GRAD)
                opt.step()

        if total_steps%10000<ROLLOUT_LEN:
            torch.save({"net":net.state_dict()},SAVE_PATH)
            print("Saved weights at",total_steps)

    torch.save({"net":net.state_dict()},SAVE_PATH)

if __name__=="__main__":
    main()

Saved weights at 10240
Saved weights at 20480
Saved weights at 30208
Saved weights at 40448
Saved weights at 50176
Saved weights at 60416
Saved weights at 70144
Saved weights at 80384
Saved weights at 90112
Saved weights at 100352
Saved weights at 110080
Saved weights at 120320
Saved weights at 130048
Saved weights at 140288
Saved weights at 150016


In [3]:

import os, math, random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ── CONFIG ────────────────────────────────────────────────────────────
MODE = "run"   # "run" | "file" | "manual"

# if MODE == "file": path to .npy file with shape (N, 2) — (step, reward)
LOG_FILE = "training_scores.npy"

# if MODE == "manual": paste list of episode rewards from your Colab output
MANUAL_SCORES = []   # e.g. [-980, -1020, -850, ...]

# instrumented run length (only used if MODE == "run")
PLOT_STEPS = 200_000

plt.rcParams.update({
    "font.family": "DejaVu Serif", "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3, "figure.dpi": 150,
})

# ── network (matches your submission) ────────────────────────────────
STATE_DIM=18; N_ACTIONS=5; GRU_H=128; FC_E=64
ACTIONS=("L45","L22","FW","R22","R45")
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed=nn.Sequential(nn.Linear(STATE_DIM,FC_E),nn.Tanh(),nn.Linear(FC_E,FC_E),nn.Tanh())
        self.gru=nn.GRUCell(FC_E,GRU_H)
        self.actor=nn.Sequential(nn.Linear(GRU_H,FC_E),nn.Tanh(),nn.Linear(FC_E,N_ACTIONS))
        self.critic=nn.Sequential(nn.Linear(GRU_H,FC_E),nn.Tanh(),nn.Linear(GRU_H if False else FC_E,1))
        # fix critic head
        self.critic=nn.Sequential(nn.Linear(GRU_H,FC_E),nn.Tanh(),nn.Linear(FC_E,1))
        for m in self.modules():
            if isinstance(m,nn.Linear):
                nn.init.orthogonal_(m.weight,gain=math.sqrt(2)); nn.init.zeros_(m.bias)
        nn.init.orthogonal_(self.actor[-1].weight,gain=0.01)
        nn.init.orthogonal_(self.critic[-1].weight,gain=1.0)
    def forward(self,obs,h):
        h=self.gru(self.embed(obs),h)
        return self.actor(h),self.critic(h).squeeze(-1),h
    def init_h(self,b=1): return torch.zeros(b,GRU_H,device=device)

# ── training run with score logging ──────────────────────────────────
from obelix import OBELIX

ARENA=500; SCALE=5; MAX_EP=1000; WALL=True; SPD=2; SEED=42
DIFF_W={0:0.3,2:0.3,3:0.4}
LR=2.5e-4; GAMMA=0.99; GAE_LAM=0.95; CLIP=0.2
ENT_C=0.01; VF_C=0.5; GRAD=0.5; EPOCHS=4; CHUNK=32; ROLL=512

_diffs=list(DIFF_W); _dp=np.array([DIFF_W[d] for d in _diffs]); _dp/=_dp.sum()

def gae(rews,vals,dones,lv):
    n=len(rews); adv=np.zeros(n,np.float32); g=0.0
    for t in reversed(range(n)):
        nxt=lv if t==n-1 else vals[t+1]
        d=rews[t]+GAMMA*nxt*(1-dones[t])-vals[t]; g=d+GAMMA*GAE_LAM*(1-dones[t])*g; adv[t]=g
    return adv,adv+vals

def run_and_log(total_steps):
    net=Net().to(device); opt=optim.Adam(net.parameters(),lr=LR,eps=1e-5)
    Bo=np.zeros((ROLL,STATE_DIM),np.float32); Ba=np.zeros(ROLL,np.int64)
    Blp=np.zeros(ROLL,np.float32); Bv=np.zeros(ROLL,np.float32)
    Br=np.zeros(ROLL,np.float32); Bd=np.zeros(ROLL,np.float32)
    Bh=np.zeros((ROLL,GRU_H),np.float32)

    d=int(np.random.choice(_diffs,p=_dp))
    ep=0
    env=OBELIX(scaling_factor=SCALE,arena_size=ARENA,max_steps=MAX_EP,
               wall_obstacles=WALL,difficulty=d,box_speed=SPD,seed=SEED)
    obs=env.reset(seed=SEED); h=net.init_h(1)
    steps=0; ep_r=0.0; log=[]   # list of (step, ep_reward)

    while steps<total_steps:
        net.eval()
        with torch.no_grad():
            for t in range(ROLL):
                obs_t=torch.FloatTensor(obs).unsqueeze(0).to(device)
                logits,v,h_new=net(obs_t,h); dist=Categorical(logits=logits); act=dist.sample()
                ai=act.item(); obs_n,raw,done=env.step(ACTIONS[ai],render=False)
                Bo[t]=obs; Ba[t]=ai; Blp[t]=dist.log_prob(act).item()
                Bv[t]=v.item(); Br[t]=raw; Bd[t]=float(done)
                Bh[t]=h.squeeze(0).cpu().numpy()
                obs=obs_n; h=h_new; ep_r+=raw; steps+=1
                if done:
                    log.append((steps,ep_r)); ep_r=0.0; ep+=1
                    d=int(np.random.choice(_diffs,p=_dp))
                    env=OBELIX(scaling_factor=SCALE,arena_size=ARENA,max_steps=MAX_EP,
                               wall_obstacles=WALL,difficulty=d,box_speed=SPD,seed=SEED+ep)
                    obs=env.reset(seed=SEED+ep); h=net.init_h(1)

        with torch.no_grad():
            _,lv,_=net(torch.FloatTensor(obs).unsqueeze(0).to(device),h); lv=lv.item()
        adv,ret=gae(Br,Bv,Bd,lv)
        net.train()
        adv_t=torch.FloatTensor(adv).to(device); adv_t=(adv_t-adv_t.mean())/(adv_t.std()+1e-8)
        ret_t=torch.FloatTensor(ret).to(device); lp_t=torch.FloatTensor(Blp).to(device)
        a_t=torch.LongTensor(Ba).to(device)
        for _ in range(EPOCHS):
            starts=list(range(0,ROLL-CHUNK,CHUNK)); random.shuffle(starts)
            for s in starts:
                h_c=torch.FloatTensor(Bh[s]).unsqueeze(0).to(device)
                nlp,nv,ne=[],[],[]
                for t2 in range(s,s+CHUNK):
                    logits,v2,h_c=net(torch.FloatTensor(Bo[t2]).unsqueeze(0).to(device),h_c)
                    dist2=Categorical(logits=logits)
                    nlp.append(dist2.log_prob(a_t[t2].unsqueeze(0)).squeeze())
                    nv.append(v2.squeeze()); ne.append(dist2.entropy().squeeze())
                    if Bd[t2]: h_c=net.init_h(1)
                nlp=torch.stack(nlp); nv=torch.stack(nv); ne=torch.stack(ne)
                sl=slice(s,s+CHUNK); r2=(nlp-lp_t[sl]).exp()
                loss=(-torch.min(r2*adv_t[sl],torch.clamp(r2,1-CLIP,1+CLIP)*adv_t[sl]).mean()
                     +VF_C*nn.functional.mse_loss(nv,ret_t[sl])-ENT_C*ne.mean())
                opt.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(net.parameters(),GRAD); opt.step()

        if steps%20000<ROLL:
            n_ep=len(log); mean=np.mean([x[1] for x in log[-20:]]) if log else 0
            print(f"steps={steps:>7,}  ep={n_ep:4d}  mean20={mean:8.1f}")

    return log

# ── plotting ─────────────────────────────────────────────────────────
def moving_avg(arr, w):
    if len(arr)<w: w=max(1,len(arr)//3)
    return np.convolve(arr,np.ones(w)/w,mode="valid")

def make_plot(log, out="figures/training_curves.pdf"):
    steps=np.array([x[0] for x in log]); rews=np.array([x[1] for x in log])
    os.makedirs("figures",exist_ok=True)

    w50 =min(50, max(1,len(rews)//10))
    w200=min(200,max(1,len(rews)//4))
    ma50 =moving_avg(rews,w50)
    ma200=moving_avg(rews,w200)
    xs50 =steps[len(steps)-len(ma50):]
    xs200=steps[len(steps)-len(ma200):]

    fig,axes=plt.subplots(1,2,figsize=(10,3.8))

    # left: raw + smoothed
    axes[0].plot(steps,rews,alpha=0.18,color="#4878D0",lw=0.6,label="raw")
    axes[0].plot(xs50, ma50, color="#4878D0",lw=1.2,label=f"MA-{w50}")
    axes[0].plot(xs200,ma200,color="#1a3e8a",lw=1.8,label=f"MA-{w200}")
    axes[0].set_xlabel("Environment steps"); axes[0].set_ylabel("Episode reward")
    axes[0].set_title("Training reward over time")
    axes[0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f"{x/1e6:.1f}M" if x>=1e6 else f"{int(x/1e3)}k"))
    axes[0].legend(fontsize=8)

    # right: rolling mean with std band
    w_band=min(100,max(5,len(rews)//8))
    def rolling_stats(arr,w):
        means,stds=[],[]
        for i in range(w,len(arr)+1):
            chunk=arr[i-w:i]; means.append(np.mean(chunk)); stds.append(np.std(chunk))
        return np.array(means),np.array(stds)
    rmean,rstd=rolling_stats(rews,w_band)
    xs_band=steps[w_band-1:]
    axes[1].plot(xs_band,rmean,color="#EE854A",lw=1.8,label=f"Rolling mean (w={w_band})")
    axes[1].fill_between(xs_band,rmean-rstd,rmean+rstd,alpha=0.2,color="#EE854A")
    axes[1].axhline(-1000,color="gray",lw=0.8,ls="--",label="No-success baseline")
    axes[1].set_xlabel("Environment steps"); axes[1].set_ylabel("Episode reward")
    axes[1].set_title("Rolling mean with std deviation")
    axes[1].xaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f"{x/1e6:.1f}M" if x>=1e6 else f"{int(x/1e3)}k"))
    axes[1].legend(fontsize=8)

    fig.tight_layout(); fig.savefig(out); plt.close(fig)
    print(f"Saved {out}")

    # also save raw data for later use
    np.save("training_scores.npy", np.array(log))
    print("Raw scores saved to training_scores.npy")

# ── main ─────────────────────────────────────────────────────────────
def main():
    if MODE=="file":
        data=np.load(LOG_FILE); log=[(int(r[0]),float(r[1])) for r in data]
        print(f"Loaded {len(log)} episodes from {LOG_FILE}")
    elif MODE=="manual":
        if not MANUAL_SCORES:
            raise ValueError("Set MANUAL_SCORES list or change MODE")
        log=list(enumerate(MANUAL_SCORES))
    else:
        print(f"Running {PLOT_STEPS:,} instrumented training steps on {device}...")
        log=run_and_log(PLOT_STEPS)

    print(f"Total episodes: {len(log)}")
    if log:
        print(f"First 5 rewards: {[round(x[1],1) for x in log[:5]]}")
        print(f"Last 5 rewards:  {[round(x[1],1) for x in log[-5:]]}")
        print(f"Best episode: {max(x[1] for x in log):.1f}")
    make_plot(log)

if __name__=="__main__":
    main()

Running 200,000 instrumented training steps on cpu...
steps= 20,480  ep=  20  mean20=-12621.0
steps= 40,448  ep=  41  mean20=-15890.5
steps= 60,416  ep=  63  mean20=-26775.2
steps= 80,384  ep=  83  mean20=-21872.8
steps=100,352  ep= 103  mean20=-25828.0
steps=120,320  ep= 123  mean20=-28616.5
steps=140,288  ep= 144  mean20=-17161.6
steps=160,256  ep= 164  mean20=-17988.8
steps=180,224  ep= 188  mean20=-26405.7
steps=200,192  ep= 208  mean20=-33603.9
Total episodes: 208
First 5 rewards: [-973.0, -972.0, -21570.0, -974.0, -20367.0]
Last 5 rewards:  [-21567.0, -51578.0, -971.0, -37967.0, -3382.0]
Best episode: 2023.0
Saved figures/training_curves.pdf
Raw scores saved to training_scores.npy


In [4]:
import numpy as np
import matplotlib.pyplot as plt

LOG_FILE="training_scores.npy"
OUT_FILE="reward_vs_episode.png"

data=np.load(LOG_FILE,allow_pickle=True)

if data.ndim==2 and data.shape[1]>=2:
    episodes=np.arange(1,len(data)+1)
    rewards=data[:,1].astype(float)
else:
    rewards=np.array(data,dtype=float).reshape(-1)
    episodes=np.arange(1,len(rewards)+1)

def ma(x,w):
    w=max(1,min(w,len(x)))
    return np.convolve(x,np.ones(w)/w,mode="valid")

plt.figure(figsize=(10,5))
plt.plot(episodes,rewards,alpha=0.25,lw=0.8,label="raw")

w1=max(5,len(rewards)//20)
w2=max(10,len(rewards)//10)

plt.plot(np.arange(w1,len(rewards)+1),ma(rewards,w1),lw=2,label=f"MA-{w1}")
plt.plot(np.arange(w2,len(rewards)+1),ma(rewards,w2),lw=2,label=f"MA-{w2}")

plt.xlabel("Episode")
plt.ylabel("Reward")
plt.title("Reward vs Episode")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(OUT_FILE,dpi=200)
plt.show()

print("saved:",OUT_FILE)

saved: reward_vs_episode.png
